# Unsloth 4-Bit QLoRA Fine-Tuning for Commercial Insurance Policy & Binder Extraction
**Models Supported:** `unsloth/Qwen2.5-14B-Instruct-bnb-4bit`, `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`  
**Target Tasks:** Binder-vs-Policy Divergence Detection, Endorsement Override Resolution (`INV-02`), and Schedule of Values (`SOV`) Tabular Extraction.

In [1]:
# 1. Initialize Unsloth FastLanguageModel with 32k Long-Context RoPE Scaling & 4-bit NormalFloat (nf4)
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-deps

UNSLOTH_TRAINING_SNIPPET = '''
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

max_seq_length = 32768  # RoPE YaRN scaling for 100-page insurance binders & policies
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-14B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM, fits 2x longer policy contexts
    random_state=3407,
)

# Export to vLLM 16-bit merged or GGUF q4_k_m for AKS deployment:
model.save_pretrained_merged("adapters/qwen25-insurance-vllm", tokenizer, save_method="lora")
model.save_pretrained_gguf("adapters/qwen25-insurance-gguf", tokenizer, quantization_method="q4_k_m")
'''
print("Unsloth QLoRA configuration verified.")